In [ ]:
import yaml
import polars as pl
import numpy as np

In [ ]:
config_path = '../config_wgs_cadd.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

len(all_annotation_list)

In [ ]:
ann = pl.read_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/wgs_78_genes_cadd_annotations.parquet')
ann

In [ ]:

def check_positive_negative(df: pl.DataFrame, columns):
    results = {}
    for col in columns:
        if col in df.columns:
            non_null = df.select(pl.col(col).drop_nulls())[col]
            if non_null.is_empty():
                results[col] = False  # Only nulls
            else:
                min_val = non_null.min()
                max_val = non_null.max()
                results[col] = (min_val < 0) and (max_val > 0)
        else:
            results[col] = False  # Column not found
    return results

# Example usage:
columns_to_check = all_annotation_list
positive_negative_check = check_positive_negative(ann, columns_to_check)

# Print the results
for column, has_both in positive_negative_check.items():
    if has_both:
        print(f"Column '{column}': Contains both positive and negative values.")

In [ ]:
def split_pos_neg_lazy(df: pl.LazyFrame, columns):
    # Start with the lazy frame
    lf = df

    for col in columns:
        if col in df.columns:
            pos_col = (
                pl.when(pl.col(col) > 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_pos")
            )

            neg_col = (
                pl.when(pl.col(col) < 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_neg")
            )

            lf = lf.with_columns([pos_col, neg_col])

    return lf

In [ ]:
# Get the columns that contain both positive and negative values
mix_cols = [k for k, v in positive_negative_check.items() if v]

# Split the positive and negative values into separate columns
split_ann = split_pos_neg_lazy(ann.lazy(), mix_cols).collect()
split_ann

In [ ]:
diff_cols = set(split_ann.columns) - set(ann.columns)
len(diff_cols)

In [ ]:
min_max_values = {}

for col in diff_cols:
    min_val = split_ann[col].min()
    max_val = split_ann[col].max()
    min_max_values[col] = {"min": min_val, "max": max_val}

for col, values in min_max_values.items():
    print(f"Column: {col}, Min: {values['min']}, Max: {values['max']}")

In [ ]:
split_ann.lazy().sink_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/wgs_78_genes_cadd_annotations_posnegsplit.parquet')

In [ ]:
ca = pl.scan_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/cadd_vep_annotations_processed_final_selected.parquet')
ca.head().collect()